<a href="https://colab.research.google.com/github/ulugbek-lab/ClarityLM/blob/main/ClarityLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch bitsandbytes datasets peft trl

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print(tokenizer.special_tokens_map)
# print(tokenizer.vocab_size)


# Quantization:

# store model weights using fewer bits
# e.g. 16-bit weights → 4-bit weights
# significantly reduces GPU memory usage
# 4K => 1080p

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load supported model weights in a 4-bit quantized representation instead of normal 16/32-bit storage.
    bnb_4bit_use_double_quant=True, # When you quantize model weights, you need some extra numbers describing how to map compressed values back toward their approximate real values.
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4' # It is a 4-bit data representation designed to work well with neural-network weights whose values tend to follow roughly normal-like distributions.
)

bnb_config


model = AutoModelForCausalLM.from_pretrained( # Causal Language Model means a model trained to predict tokens autoregressively: previous tokens → next token
    model_id, # Now you load the ACTUAL LLM
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    dtype=torch.bfloat16
)



In [ ]:
# Before fine-tuning
prompt = 'explain what is linear regression?'
# prompt =' siberian husky is type of dog that'

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) #GPU CPU; todevice -> Move my input tensors to the same main device as the model.

with torch.no_grad():
  output_ids = model.generate( # is performing this loop for you. it predicts next token
      **inputs,  # **unpacking => passes both input_ids and attention_mask.
      max_new_tokens=80,
      do_sample=True, # Randomly sample from the probability distribution instead of always taking the exact highest-probability option.
      temperature=0.7 # Temperature modifies the shape of the next-token probability distribution before sampling.
      # low temperature more predictable , high temperature , more diverse and random


  )


  print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

  print(model)

  first_block = model.model.layers[0]
  print("first_block:------------------------------",first_block)
  print("=========================")
  print(first_block.self_attn)
  print("=========================")
  print(model.config)

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"total p (including frozen 4-bits): {total_params:,}")
# input_ids
#    ↓
# embedding layer   ← HERE
#    ↓
# embedding vectors
#    ↓
# Transformer layers
#    ↓
# next-token prediction

## datasets library | load_dataset
* instruction tunning

In [ ]:

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# dataset
dataset

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Intrsuctions:\n"
        f"{instruction}\n\n"
        "### input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Intrsuctions:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )

# generate_prompt(dataset[8])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

dataset[0]

In [ ]:
dataset[0]

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42) # choose random 7000 data with 42 batch

In [ ]:
# [4,5,6,4,6,7,8,2,3,5]
dataset

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory


In [ ]:


lora_config = LoraConfig(
    task_type="CAUSAL_LM", # autoregressive next-token prediction model
    r=8, # rank; the higher rank , higher params , use more momeory; 8 is average; It controls the size of those small LoRA matrices. 2080 * 8;
    lora_alpha=16, # 8, 32;  This controls the strength/scaling of the LoRA update.
    target_modules=["q_proj", "v_proj"], # query & value
    lora_dropout=0.05, # During training, roughly 5% of the relevant LoRA input activations are randomly dropped for that step.
)
#  Query = "What am I looking for?"


In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
# QLora
# Lora

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1, # Send 1 training example through each GPU/device at a time.
    gradient_accumulation_steps=4, # VRAM; Process 1 example at a time, but accumulate gradients from 4 mini-steps before updating the weights.
    num_train_epochs=2, # If higher, overfit. it determined after training/ for dataset
    logging_steps=30,
    output_dir="./clarityLM",
    save_strategy="epoch",
    bf16=True,   # Disable bf16 since T4 GPU doesn't support it
    fp16=False,    # Enable fp16 for T4 GPU acceleration
    report_to="none"
)

In [ ]:
# dataset['text']
print(dataset.column_names)

In [ ]:
# SFTTrainer

trainer = SFTTrainer(
    model=model, # TinyLlama/TinyLlama-1.1B-Chat-v1.0
    train_dataset=dataset, # Alpaca
    formatting_func=lambda x: x['text'],  # Give me an object x, and I'll return its "text" value. no "category" only "text"
    args=training_args # training argument
)
dataset['text']

trainer.train()
model.save_pretrained('clarityLM')
tokenizer.save_pretrained('clarityLM')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)

model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

In [ ]:

prompt = 'explain what is linear regression in a simple words?'


inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
  output_base = model_base.generate(
      **inputs_base,
      max_new_tokens=120,
      do_sample=True,
      temperature=0.7

  )
  print("========== BASE MODEL OUTPUT ===========")

  print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

#FINE_TUNED MODEL INFERENCE



In [ ]:
# model_path = "clarityLM"

# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModelForCausalLM.from_pretrained(
#     model_path,
#     device_map="auto"
# )


In [ ]:

# prompt = 'explain what is linear regression in a simple words?'


# inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)

# with torch.no_grad():
#   output_ft = model_ft.generate(
#       **inputs_ft,
#       max_new_tokens=120,
#       do_sample=True,
#       temperature=0.7

#   )
#   print("========== FINE_TUNED MODEL OUTPUT ===========")

#   print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

In [ ]:
!zip -r clarityLM.zip clarityLM

In [ ]:
from google.colab import files
files.dowload("clarityLM.zip")

In [ ]:
################################################
#                MODEL TESTING                 #
################################################

In [ ]:
!pip install -q langdetect

In [ ]:
!unzip clarityLM.zip

In [ ]:
!pip install -U torchao

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
import torch

base_model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
adapter_dir = "/content/clarityLM"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

In [ ]:
from langdetect import detect_langs

def ask_clarityLM(prompt:str, max_new_tokens=256):
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Lowered confidence threshold and removed the length check to block short non-English inputs
    if lang_code != "en" and confidence >= 0.50:
        return "Sorry, you should ask a question only in english."

    # Automatically use GPU if available, otherwise fallback to CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

In [ ]:
ask_clarityLM('tell me about python')